# PART A

In [137]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("garments_worker_productivity.csv")

In [3]:
print("Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Shape: (1197, 16)

Data Types:
date                         str
quarter                      str
department                   str
day                          str
team                       int64
targeted_productivity    float64
smv                      float64
wip                      float64
over_time                  int64
incentive                  int64
idle_time                float64
idle_men                   int64
no_of_style_change         int64
no_of_workers            float64
actual_productivity      float64
MeetsTarget                int64
dtype: object

Missing Values:
date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actu

In [4]:
print("\nClassification Target:")
print(df["MeetsTarget"].value_counts())

print("\nClassification Target (%):")
print(df["MeetsTarget"].value_counts(normalize=True) * 100)


Classification Target:
MeetsTarget
1    875
0    322
Name: count, dtype: int64

Classification Target (%):
MeetsTarget
1    73.099415
0    26.900585
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split

# Create row indices
indices = np.arange(len(df))

# Fixed train-test split
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=df["MeetsTarget"]
)

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))

Training samples: 957
Testing samples: 240


In [7]:
print("Overall:")
print(df["MeetsTarget"].value_counts(normalize=True))

print("\nTraining:")
print(df.loc[train_idx, "MeetsTarget"].value_counts(normalize=True))

print("\nTesting:")
print(df.loc[test_idx, "MeetsTarget"].value_counts(normalize=True))

Overall:
MeetsTarget
1    0.730994
0    0.269006
Name: proportion, dtype: float64

Training:
MeetsTarget
1    0.731452
0    0.268548
Name: proportion, dtype: float64

Testing:
MeetsTarget
1    0.729167
0    0.270833
Name: proportion, dtype: float64


In [8]:
feature_columns = [
    "quarter",
    "department",
    "day",
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]

In [10]:
X_reg = df[feature_columns]
y_reg = df["actual_productivity"]

In [11]:
X_cls = df[feature_columns]
y_cls = df["MeetsTarget"]

In [12]:
categorical_features = [
    "quarter",
    "department",
    "day"
]

numerical_features = [
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [14]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [15]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [16]:
X_train_reg = X_reg.iloc[train_idx]
X_test_reg = X_reg.iloc[test_idx]

y_train_reg = y_reg.iloc[train_idx]
y_test_reg = y_reg.iloc[test_idx]

In [17]:
from sklearn.linear_model import LinearRegression

In [19]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

In [20]:
import time

In [21]:
start_time = time.perf_counter()

linear_model.fit(X_train_reg, y_train_reg)

linear_train_time = time.perf_counter() - start_time

print("Linear Regression training time:",
      linear_train_time, "seconds")

Linear Regression training time: 0.049459200000455894 seconds


In [22]:
start_time = time.perf_counter()

y_pred_reg = linear_model.predict(X_test_reg)

linear_prediction_time = time.perf_counter() - start_time

print("Linear Regression prediction time:",
      linear_prediction_time, "seconds")

Linear Regression prediction time: 0.016374699999687437 seconds


In [23]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [25]:
mae = mean_absolute_error(y_test_reg, y_pred_reg)

rmse = np.sqrt(
    mean_squared_error(y_test_reg, y_pred_reg)
)

r2 = r2_score(y_test_reg, y_pred_reg)

In [138]:
print("\nLinear Regression Results")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)
print("Training Time   :", linear_train_time)
print("Prediction Time :", linear_prediction_time)


Linear Regression Results
MAE : 0.10727843302917223
RMSE: 0.1437402979105884
R²  : 0.2885214314434412
Training Time   : 0.049459200000455894
Prediction Time : 0.016374699999687437


In [29]:
# CLASSIFICATION
X_cls = df[feature_columns]
y_cls = df["MeetsTarget"]

X_train_cls = X_cls.iloc[train_idx]
X_test_cls = X_cls.iloc[test_idx]

y_train_cls = y_cls.iloc[train_idx]
y_test_cls = y_cls.iloc[test_idx]

In [30]:
from sklearn.linear_model import LogisticRegression

In [31]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [32]:
start_time = time.perf_counter()

logistic_model.fit(X_train_cls, y_train_cls)

logistic_train_time = time.perf_counter() - start_time

print("Logistic Regression training time:",
      logistic_train_time, "seconds")

Logistic Regression training time: 0.058267000000341795 seconds


In [33]:
start_time = time.perf_counter()

y_pred_cls = logistic_model.predict(X_test_cls)

logistic_prediction_time = time.perf_counter() - start_time

print("Logistic Regression prediction time:",
      logistic_prediction_time, "seconds")

Logistic Regression prediction time: 0.015083100000083505 seconds


In [35]:
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score)

In [36]:
accuracy = accuracy_score(y_test_cls, y_pred_cls)

precision = precision_score(y_test_cls, y_pred_cls)

recall = recall_score(y_test_cls, y_pred_cls)

f1 = f1_score(y_test_cls, y_pred_cls)

In [37]:
print("\nLogistic Regression Results")
print("---------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)
print("Training Time   :", logistic_train_time)
print("Prediction Time :", logistic_prediction_time)


Logistic Regression Results
---------------------------
Accuracy : 0.7083333333333334
Precision: 0.7611940298507462
Recall   : 0.8742857142857143
F1-score : 0.8138297872340425
Training Time   : 0.058267000000341795
Prediction Time : 0.015083100000083505


# PART B

In [38]:
scratch_df = df.copy()

In [40]:
X_scratch = scratch_df[feature_columns].copy()

In [41]:
y_reg_scratch = scratch_df["actual_productivity"].to_numpy()

y_cls_scratch = scratch_df["MeetsTarget"].to_numpy()

In [42]:
X_train_scratch = X_scratch.iloc[train_idx].copy()
X_test_scratch = X_scratch.iloc[test_idx].copy()

y_train_reg_scratch = y_reg_scratch[train_idx]
y_test_reg_scratch = y_reg_scratch[test_idx]

y_train_cls_scratch = y_cls_scratch[train_idx]
y_test_cls_scratch = y_cls_scratch[test_idx]

In [44]:
numerical_features = [
    "team",
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers"
]

In [45]:
train_medians = X_train_scratch[numerical_features].median()

In [48]:
print(train_medians)

team                        6.00
targeted_productivity       0.75
smv                        15.26
wip                      1038.00
over_time                4080.00
incentive                   0.00
idle_time                   0.00
idle_men                    0.00
no_of_style_change          0.00
no_of_workers              34.00
dtype: float64


In [49]:
X_train_scratch[numerical_features] = (
    X_train_scratch[numerical_features]
    .fillna(train_medians)
)

X_test_scratch[numerical_features] = (
    X_test_scratch[numerical_features]
    .fillna(train_medians)
)

In [50]:
train_mean = X_train_scratch[numerical_features].mean()
train_std = X_train_scratch[numerical_features].std()

In [51]:
X_train_scratch[numerical_features] = (
    X_train_scratch[numerical_features] - train_mean
) / train_std

In [52]:
X_test_scratch[numerical_features] = (
    X_test_scratch[numerical_features] - train_mean
) / train_std

In [53]:
categorical_features = [
    "quarter",
    "department",
    "day"
]

In [54]:
combined_cat = pd.concat([
    X_train_scratch[categorical_features],
    X_test_scratch[categorical_features]
])

combined_cat = pd.get_dummies(
    combined_cat,
    columns=categorical_features,
    dtype=float
)

In [55]:
X_train_cat = combined_cat.iloc[:len(X_train_scratch)].copy()

X_test_cat = combined_cat.iloc[len(X_train_scratch):].copy()

In [56]:
print(X_train_cat.shape)
print(X_test_cat.shape)

(957, 14)
(240, 14)


In [57]:
X_train_num = X_train_scratch[numerical_features].astype(float)
X_test_num = X_test_scratch[numerical_features].astype(float)

In [58]:
X_train_processed = pd.concat(
    [X_train_num.reset_index(drop=True),
     X_train_cat.reset_index(drop=True)],
    axis=1
)

X_test_processed = pd.concat(
    [X_test_num.reset_index(drop=True),
     X_test_cat.reset_index(drop=True)],
    axis=1
)

In [59]:
X_train_final = X_train_processed.to_numpy(dtype=float)
X_test_final = X_test_processed.to_numpy(dtype=float)

In [60]:
print("Training shape:", X_train_final.shape)
print("Testing shape :", X_test_final.shape)

Training shape: (957, 24)
Testing shape : (240, 24)


In [61]:
X_train_lr = np.c_[
    np.ones(X_train_final.shape[0]),
    X_train_final
]

X_test_lr = np.c_[
    np.ones(X_test_final.shape[0]),
    X_test_final
]

print("Training shape:", X_train_lr.shape)
print("Testing shape :", X_test_lr.shape)

Training shape: (957, 25)
Testing shape : (240, 25)


In [62]:
import time

start_time = time.perf_counter()

theta_lr = (
    np.linalg.pinv(X_train_lr.T @ X_train_lr)
    @ X_train_lr.T
    @ y_train_reg_scratch
)

scratch_linear_train_time = time.perf_counter() - start_time

print("Training time:",
      scratch_linear_train_time,
      "seconds")

Training time: 0.017325099999652593 seconds


In [63]:
print("Number of parameters:", len(theta_lr))

Number of parameters: 25


In [64]:
start_time = time.perf_counter()

y_pred_reg_scratch = X_test_lr @ theta_lr

scratch_linear_prediction_time = time.perf_counter() - start_time

print("Prediction time:",
      scratch_linear_prediction_time,
      "seconds")

Prediction time: 0.00011230000018258579 seconds


In [66]:
def mean_absolute_error_manual(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

In [67]:
def root_mean_squared_error_manual(y_true, y_pred):
    return np.sqrt(
        np.mean((y_true - y_pred) ** 2)
    )

In [68]:
def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    return 1 - (ss_res / ss_tot)

In [70]:
scratch_mae = mean_absolute_error_manual(
    y_test_reg_scratch,
    y_pred_reg_scratch
)

scratch_rmse = root_mean_squared_error_manual(
    y_test_reg_scratch,
    y_pred_reg_scratch
)

scratch_r2 = r2_score_manual(
    y_test_reg_scratch,
    y_pred_reg_scratch
)

In [72]:
print("\nScratch Linear Regression Results")
print("MAE :", scratch_mae)
print("RMSE:", scratch_rmse)
print("R²  :", scratch_r2)
print("Training Time   :", scratch_linear_train_time)
print("Prediction Time :", scratch_linear_prediction_time)


Scratch Linear Regression Results
MAE : 0.10727843302917249
RMSE: 0.14374029791058823
R²  : 0.28852143144344267
Training Time   : 0.017325099999652593
Prediction Time : 0.00011230000018258579


In [73]:
train_cat = pd.get_dummies(
    X_train_scratch[categorical_features],
    columns=categorical_features,
    dtype=float
)

In [74]:
test_cat = pd.get_dummies(
    X_test_scratch[categorical_features],
    columns=categorical_features,
    dtype=float
)

In [75]:
test_cat = test_cat.reindex(
    columns=train_cat.columns,
    fill_value=0
)

In [77]:
print("Training categorical shape:", train_cat.shape)
print("Testing categorical shape :", test_cat.shape)

Training categorical shape: (957, 14)
Testing categorical shape : (240, 14)


In [79]:
X_train_num = X_train_scratch[numerical_features].astype(float)
X_test_num = X_test_scratch[numerical_features].astype(float)

X_train_processed = pd.concat(
    [
        X_train_num.reset_index(drop=True),
        train_cat.reset_index(drop=True)
    ],
    axis=1
)

X_test_processed = pd.concat(
    [
        X_test_num.reset_index(drop=True),
        test_cat.reset_index(drop=True)
    ],
    axis=1
)

X_train_final = X_train_processed.to_numpy(dtype=float)
X_test_final = X_test_processed.to_numpy(dtype=float)

In [80]:
print("Training shape:", X_train_final.shape)
print("Testing shape :", X_test_final.shape)

Training shape: (957, 24)
Testing shape : (240, 24)


In [81]:
# LOGISTIC REGRESSION
X_train_log = np.c_[
    np.ones(X_train_final.shape[0]),
    X_train_final
]

X_test_log = np.c_[
    np.ones(X_test_final.shape[0]),
    X_test_final
]

print("Training shape:", X_train_log.shape)
print("Testing shape :", X_test_log.shape)

Training shape: (957, 25)
Testing shape : (240, 25)


In [83]:
y_train_cls_scratch
y_test_cls_scratch

array([0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0,
       0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0])

In [84]:
y_train_log = y_train_cls_scratch.astype(float)
y_test_log = y_test_cls_scratch.astype(float)

In [85]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [86]:
test_values = np.array([-10, -2, 0, 2, 10])

print(sigmoid(test_values))

[4.53978687e-05 1.19202922e-01 5.00000000e-01 8.80797078e-01
 9.99954602e-01]


In [87]:
weights = np.zeros(X_train_log.shape[1])

print("Number of weights:", len(weights))

Number of weights: 25


In [112]:
def train_logistic_regression(
    X,
    y,
    learning_rate=0.01,
    iterations=10000
):
    n_samples, n_features = X.shape

    weights = np.zeros(n_features)

    for i in range(iterations):

        # Linear combination
        z = X @ weights

        # Predicted probabilities
        probabilities = sigmoid(z)

        # Gradient
        gradient = (
            X.T @ (probabilities - y)
        ) / n_samples

        # Update weights
        weights = weights - learning_rate * gradient

    return weights

In [113]:
start_time = time.perf_counter()

logistic_weights = train_logistic_regression(
    X_train_log,
    y_train_log,
    learning_rate=0.01,
    iterations=10000
)

scratch_logistic_train_time = (
    time.perf_counter() - start_time
)

print(
    "Training time:",
    scratch_logistic_train_time,
    "seconds"
)

Training time: 0.27165059999970254 seconds


In [114]:
start_time = time.perf_counter()

test_probabilities = sigmoid(
    X_test_log @ logistic_weights
)

In [115]:
y_pred_cls_scratch = (
    test_probabilities >= 0.5
).astype(int)

scratch_logistic_prediction_time = (
    time.perf_counter() - start_time
)

print(
    "Prediction time:",
    scratch_logistic_prediction_time,
    "seconds"
)

Prediction time: 1.3441604999989067 seconds


In [116]:
def confusion_matrix_manual(y_true, y_pred):

    tp = np.sum(
        (y_true == 1) & (y_pred == 1)
    )

    tn = np.sum(
        (y_true == 0) & (y_pred == 0)
    )

    fp = np.sum(
        (y_true == 0) & (y_pred == 1)
    )

    fn = np.sum(
        (y_true == 1) & (y_pred == 0)
    )

    return tp, tn, fp, fn

In [117]:
tp, tn, fp, fn = confusion_matrix_manual(
    y_test_log,
    y_pred_cls_scratch
)

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

TP: 161
TN: 16
FP: 49
FN: 14


In [118]:
def accuracy_manual(y_true, y_pred):

    return np.mean(y_true == y_pred)

In [119]:
def precision_manual(y_true, y_pred):

    tp, tn, fp, fn = confusion_matrix_manual(
        y_true,
        y_pred
    )

    return tp / (tp + fp)

In [120]:
def recall_manual(y_true, y_pred):

    tp, tn, fp, fn = confusion_matrix_manual(
        y_true,
        y_pred
    )

    return tp / (tp + fn)

In [121]:
def f1_manual(y_true, y_pred):

    precision = precision_manual(
        y_true,
        y_pred
    )

    recall = recall_manual(
        y_true,
        y_pred
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )

In [122]:
scratch_accuracy = accuracy_manual(
    y_test_log,
    y_pred_cls_scratch
)

scratch_precision = precision_manual(
    y_test_log,
    y_pred_cls_scratch
)

scratch_recall = recall_manual(
    y_test_log,
    y_pred_cls_scratch
)

scratch_f1 = f1_manual(
    y_test_log,
    y_pred_cls_scratch
)

In [123]:
print("\nScratch Logistic Regression Results")
print("Accuracy :", scratch_accuracy)
print("Precision:", scratch_precision)
print("Recall   :", scratch_recall)
print("F1-score :", scratch_f1)
print("Training Time   :", scratch_logistic_train_time)
print("Prediction Time :", scratch_logistic_prediction_time)


Scratch Logistic Regression Results
Accuracy : 0.7375
Precision: 0.7666666666666667
Recall   : 0.92
F1-score : 0.8363636363636363
Training Time   : 0.27165059999970254
Prediction Time : 1.3441604999989067


In [111]:
# Warm-up
_ = sigmoid(X_test_log @ logistic_weights)

# Actual timing
start_time = time.perf_counter()

test_probabilities = sigmoid(
    X_test_log @ logistic_weights
)

y_pred_cls_scratch = (
    test_probabilities >= 0.5
).astype(int)

scratch_logistic_prediction_time = (
    time.perf_counter() - start_time
)

print(
    "Prediction time:",
    scratch_logistic_prediction_time,
    "seconds"
)

Prediction time: 0.00010860000111279078 seconds


# PART C

In [104]:
# OPTIMIZATION
def binary_cross_entropy(y_true, probabilities):

    epsilon = 1e-15

    probabilities = np.clip(
        probabilities,
        epsilon,
        1 - epsilon
    )

    loss = -np.mean(
        y_true * np.log(probabilities)
        +
        (1 - y_true) * np.log(1 - probabilities)
    )

    return loss

In [105]:
def train_logistic_regression_optimized(
    X,
    y,
    learning_rate=0.01,
    max_iterations=10000,
    tolerance=1e-7
):

    n_samples, n_features = X.shape

    weights = np.zeros(n_features)

    previous_loss = float("inf")

    for iteration in range(max_iterations):

        # Linear combination
        z = X @ weights

        # Probabilities
        probabilities = sigmoid(z)

        # Calculate loss
        loss = binary_cross_entropy(
            y,
            probabilities
        )

        # Gradient
        gradient = (
            X.T @ (probabilities - y)
        ) / n_samples

        # Update weights
        weights -= learning_rate * gradient

        # Check convergence
        if abs(previous_loss - loss) < tolerance:
            break

        previous_loss = loss

    return weights, iteration + 1

In [106]:
start_time = time.perf_counter()

optimized_weights, iterations_used = (
    train_logistic_regression_optimized(
        X_train_log,
        y_train_log,
        learning_rate=0.01,
        max_iterations=10000,
        tolerance=1e-7
    )
)

optimized_logistic_train_time = (
    time.perf_counter() - start_time
)

print(
    "Iterations used:",
    iterations_used
)

print(
    "Training time:",
    optimized_logistic_train_time,
    "seconds"
)

Iterations used: 10000
Training time: 0.7222101999996084 seconds


In [107]:
start_time = time.perf_counter()

optimized_probabilities = sigmoid(
    X_test_log @ optimized_weights
)

optimized_predictions = (
    optimized_probabilities >= 0.5
).astype(int)

optimized_logistic_prediction_time = (
    time.perf_counter() - start_time
)

In [108]:
optimized_accuracy = accuracy_manual(
    y_test_log,
    optimized_predictions
)

optimized_precision = precision_manual(
    y_test_log,
    optimized_predictions
)

optimized_recall = recall_manual(
    y_test_log,
    optimized_predictions
)

optimized_f1 = f1_manual(
    y_test_log,
    optimized_predictions
)

In [110]:
print("\nOptimized Logistic Regression")
print("Accuracy :", optimized_accuracy)
print("Precision:", optimized_precision)
print("Recall   :", optimized_recall)
print("F1-score :", optimized_f1)
print("Iterations:", iterations_used)
print("Training Time:", optimized_logistic_train_time)
print("Prediction Time:", optimized_logistic_prediction_time)


Optimized Logistic Regression
Accuracy : 0.7375
Precision: 0.7666666666666667
Recall   : 0.92
F1-score : 0.8363636363636363
Iterations: 10000
Training Time: 0.7222101999996084
Prediction Time: 0.00023130000045057386


In [124]:
def train_logistic_tuned(
    X,
    y,
    learning_rate=0.01,
    iterations=10000
):
    n_samples, n_features = X.shape

    weights = np.zeros(n_features)

    for i in range(iterations):

        # Linear score
        z = X @ weights

        # Probability
        probabilities = sigmoid(z)

        # Gradient
        gradient = (
            X.T @ (probabilities - y)
        ) / n_samples

        # Weight update
        weights -= learning_rate * gradient

    # Final loss only
    probabilities = sigmoid(X @ weights)

    final_loss = binary_cross_entropy(
        y,
        probabilities
    )

    return weights, final_loss

In [125]:
learning_rates = [
    0.001,
    0.005,
    0.01,
    0.05,
    0.1
]

results = []

for lr in learning_rates:

    start_time = time.perf_counter()

    weights, loss = train_logistic_tuned(
        X_train_log,
        y_train_log,
        learning_rate=lr,
        iterations=10000
    )

    train_time = time.perf_counter() - start_time

    probabilities = sigmoid(
        X_test_log @ weights
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    accuracy = accuracy_manual(
        y_test_log,
        predictions
    )

    precision = precision_manual(
        y_test_log,
        predictions
    )

    recall = recall_manual(
        y_test_log,
        predictions
    )

    f1 = f1_manual(
        y_test_log,
        predictions
    )

    results.append([
        lr,
        loss,
        accuracy,
        precision,
        recall,
        f1,
        train_time
    ])

In [126]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Learning Rate",
        "Training Loss",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Training Time"
    ]
)

print(results_df)

   Learning Rate  Training Loss  Accuracy  Precision    Recall        F1  \
0          0.001       0.517195  0.725000   0.731915  0.982857  0.839024   
1          0.005       0.491495  0.741667   0.760369  0.942857  0.841837   
2          0.010       0.484100  0.737500   0.766667  0.920000  0.836364   
3          0.050       0.480577  0.708333   0.763819  0.868571  0.812834   
4          0.100       0.480559  0.708333   0.763819  0.868571  0.812834   

   Training Time  
0       0.345816  
1       0.279624  
2       0.242716  
3       0.255879  
4       0.273880  


In [127]:
optimized_lr = 0.005

In [128]:
start_time = time.perf_counter()

optimized_logistic_weights, _ = train_logistic_tuned(
    X_train_log,
    y_train_log,
    learning_rate=0.005,
    iterations=10000
)

optimized_logistic_train_time = (
    time.perf_counter() - start_time
)

In [129]:
start_time = time.perf_counter()

optimized_probabilities = sigmoid(
    X_test_log @ optimized_logistic_weights
)

optimized_predictions = (
    optimized_probabilities >= 0.5
).astype(int)

optimized_logistic_prediction_time = (
    time.perf_counter() - start_time
)

In [136]:
optimized_accuracy = accuracy_manual(
    y_test_log,
    optimized_predictions
)

optimized_precision = precision_manual(
    y_test_log,
    optimized_predictions
)

optimized_recall = recall_manual(
    y_test_log,
    optimized_predictions
)

optimized_f1 = f1_manual(
    y_test_log,
    optimized_predictions
)

print("Optimized Logistic Regression")
print("Learning Rate:", 0.005)
print("Accuracy :", optimized_accuracy)
print("Precision:", optimized_precision)
print("Recall   :", optimized_recall)
print("F1-score :", optimized_f1)
print("Training Time:", optimized_logistic_train_time)
print("Prediction Time:", optimized_logistic_prediction_time)

Optimized Logistic Regression
Learning Rate: 0.005
Accuracy : 0.7416666666666667
Precision: 0.7603686635944701
Recall   : 0.9428571428571428
F1-score : 0.8418367346938777
Training Time: 0.19419509999897855
Prediction Time: 0.0002452000007906463


In [131]:
# OPTIMIZED LINEAR REGRESSION
start_time = time.perf_counter()

theta_lr_optimized, residuals, rank, singular_values = (
    np.linalg.lstsq(
        X_train_lr,
        y_train_reg_scratch,
        rcond=None
    )
)

optimized_linear_train_time = (
    time.perf_counter() - start_time
)

In [132]:
start_time = time.perf_counter()

y_pred_lr_optimized = (
    X_test_lr @ theta_lr_optimized
)

optimized_linear_prediction_time = (
    time.perf_counter() - start_time
)

In [133]:
optimized_linear_mae = mean_absolute_error_manual(
    y_test_reg_scratch,
    y_pred_lr_optimized
)

optimized_linear_rmse = root_mean_squared_error_manual(
    y_test_reg_scratch,
    y_pred_lr_optimized
)

optimized_linear_r2 = r2_score_manual(
    y_test_reg_scratch,
    y_pred_lr_optimized
)

In [135]:
print("\nOptimized Linear Regression")
print("MAE :", optimized_linear_mae)
print("RMSE:", optimized_linear_rmse)
print("R²  :", optimized_linear_r2)
print("Training Time:", optimized_linear_train_time)
print("Prediction Time:", optimized_linear_prediction_time)


Optimized Linear Regression
MAE : 0.10727843302917212
RMSE: 0.14374029791058843
R²  : 0.2885214314434411
Training Time: 0.007086999999955879
Prediction Time: 0.00010759999895526562
